# Robustesse statistique des résultats de l'étape 3

Ce carnet répond à trois objections qu'un lecteur exigeant ferait aux résultats de `research/resultats_risque.md` :

1. « 9 victoires sur 10 » n'est pas un test : les écarts de Sharpe et de volatilité entre les deux gestions sont-ils plus grands que le hasard d'échantillonnage ?
2. La corrélation monte en crise, mais une corrélation mesurée quand la volatilité du marché augmente monte mécaniquement (Forbes et Rigobon, 2002). Que reste-t-il après correction ?
3. Le thème se distingue-t-il du reste du S&P 500 construit de la même façon, c'est-à-dire du groupe témoin ?

Les fonctions viennent de `src/risque.py` et sont testées dans `tests/test_risque.py`. Le témoin est construit par `python -m src.construire_temoin`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src import risque
from src.construire_portefeuilles import verifier
from src.empreintes import verifier_manifeste

SORTIE = RACINE / "data" / "processed"

# Chaque fichier lu doit etre celui que son etape a produit et enregistre.
verifier(SORTIE)
verifier_manifeste(SORTIE / "risque_manifest.json", RACINE)
verifier_manifeste(SORTIE / "temoin_manifest.json", RACINE)
valeurs = pd.read_csv(SORTIE / "valeurs_portefeuilles.csv", index_col=0)
temoin = pd.read_csv(SORTIE / "temoin_valeurs.csv", index_col=0)
marche = pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0)["SPY"]

taux = pd.read_csv(RACINE / "data" / "raw" / "taux_sans_risque.csv",
                   index_col=0)["taux_annuel_pct"].reindex(valeurs.index).ffill()
sans_risque = (1 + taux / 100) ** (1 / 252) - 1

BLOC = 63        # un trimestre de seances : conserve l'agregation des jours agites
TIRAGES = 2000


def excedents(serie):
    r = serie.dropna().pct_change().dropna()
    return (r - sans_risque.reindex(r.index)).dropna()


def sharpe_annuel(x):
    return x.mean() / x.std(ddof=1) * np.sqrt(252)


def vol_annuelle(x):
    return x.std(ddof=1) * np.sqrt(252)

## 1. Les écarts entre gestion rééquilibrée et conservée

Pour chaque portefeuille, les deux gestions sont tirées **aux mêmes dates**, par blocs d'un trimestre. On obtient l'écart observé, son intervalle à 95 % et une probabilité bilatérale. Le Sharpe est calculé ici sur l'écart-type des rendements excédentaires, ce qui le décale de moins d'un millième de celui du rapport.

La règle de `P4` (section VI de `research/portefeuilles.md`) impose deux lectures : sur la période complète et depuis le 19 août 2004, où il compte au moins six entreprises. Les deux sont produites ici.

In [2]:
DEBUT_P4 = "2004-08-19"

lignes = []
for pf in [f"P{i}" for i in range(1, 11)]:
    a, b = excedents(valeurs[f"{pf}_reeq"]), excedents(valeurs[f"{pf}_cons"])
    a, b = a.align(b, join="inner")
    periodes = [("complete", a, b)]
    if pf == "P4":
        periodes.append((f"depuis_{DEBUT_P4}", a.loc[DEBUT_P4:], b.loc[DEBUT_P4:]))
    for periode, x, y in periodes:
        for nom, stat in [("sharpe", sharpe_annuel), ("volatilite", vol_annuelle)]:
            res = risque.ecart_bootstrap(x.to_numpy(), y.to_numpy(), stat, BLOC, TIRAGES)
            lignes.append({"portefeuille": pf, "periode": periode, "mesure": nom, **res})

gestion = pd.DataFrame(lignes)
gestion["significatif_5pct"] = gestion.p_bilaterale < 0.05
gestion.to_csv(SORTIE / "risque_bootstrap_gestion.csv", index=False, encoding="utf-8")

complete = gestion[gestion.periode == "complete"]
for mesure in ["volatilite", "sharpe"]:
    bloc = complete[complete.mesure == mesure].set_index("portefeuille")
    print(f"=== ecart reequilibre - conserve : {mesure}, periode complete")
    print(bloc[["ecart", "bas", "haut", "p_bilaterale", "significatif_5pct"]].round(4).to_string())
    print("significatifs a 5 % :", int(bloc.significatif_5pct.sum()), "sur 10")
    print()

print("=== P4, deux lectures")
print(gestion[gestion.portefeuille == "P4"].set_index(["periode", "mesure"])
      [["ecart", "bas", "haut", "p_bilaterale", "significatif_5pct"]].round(4).to_string())

=== ecart reequilibre - conserve : volatilite, periode complete
               ecart     bas    haut  p_bilaterale  significatif_5pct
portefeuille                                                         
P1           -0.0179 -0.0284 -0.0082         0.000               True
P2           -0.0174 -0.0273 -0.0079         0.000               True
P3           -0.0454 -0.0592 -0.0327         0.000               True
P4           -0.0321 -0.0497 -0.0135         0.000               True
P5           -0.0255 -0.0365 -0.0150         0.000               True
P6           -0.0059 -0.0115 -0.0013         0.002               True
P7           -0.0080 -0.0132 -0.0036         0.000               True
P8            0.0091 -0.0034  0.0247         0.166              False
P9           -0.0079 -0.0229  0.0041         0.235              False
P10          -0.0420 -0.0547 -0.0302         0.000               True
significatifs a 5 % : 8 sur 10

=== ecart reequilibre - conserve : sharpe, periode complete
    

## 2. La corrélation de crise, corrigée de la hausse de variance du marché

La correction de Forbes et Rigobon divise la corrélation de tension par $\sqrt{1 + \delta (1-\rho^2)}$, où $\delta$ est la hausse relative de la variance de `SPY` entre période calme et période de tension. Elle est ici appliquée à la corrélation moyenne entre titres, ce qui en fait une approximation : la correction exacte porte sur une paire.

In [3]:
episodes = pd.read_csv(SORTIE / "episodes_tension.csv")
tension = pd.Series(False, index=valeurs.index)
for e in episodes.itertuples():
    tension.loc[e.debut:e.creux] = True

r_marche = marche.pct_change()
variance_tension = r_marche[tension].var()
variance_calme = r_marche[~tension].var()
delta = variance_tension / variance_calme - 1
print("variance de SPY : x%.2f en tension, delta = %.3f" % (variance_tension / variance_calme, delta))

conditionnel = pd.read_csv(SORTIE / "risque_correlation_conditionnelle.csv", index_col=0)
temoin_corr = pd.read_csv(SORTIE / "temoin_correlations.csv", index_col=0)
base = pd.concat([conditionnel[["n", "rho_calme", "rho_tension"]],
                  temoin_corr.loc[["T1"], ["titres", "rho_calme", "rho_tension"]]
                  .rename(columns={"titres": "n"})])

base["rho_tension_ajustee"] = [risque.forbes_rigobon(r, delta) for r in base.rho_tension]
base["hausse_brute"] = base.rho_tension - base.rho_calme
base["hausse_ajustee"] = base.rho_tension_ajustee - base.rho_calme
base["n_eff_calme"] = [risque.n_effectif(r, n) for r, n in zip(base.rho_calme, base.n)]
base["n_eff_tension_ajuste"] = [risque.n_effectif(r, n) for r, n in zip(base.rho_tension_ajustee, base.n)]
base.to_csv(SORTIE / "risque_correlation_ajustee.csv", encoding="utf-8")
print(base.round(4).to_string())
print()
print("hausse encore positive apres correction :", int((base.hausse_ajustee > 0).sum()), "sur", len(base))

variance de SPY : x3.73 en tension, delta = 2.731
       n  rho_calme  rho_tension  rho_tension_ajustee  hausse_brute  hausse_ajustee  n_eff_calme  n_eff_tension_ajuste
P1   135     0.3069       0.4326               0.2411        0.1258         -0.0658       3.2053                4.0532
P2   111     0.3260       0.4625               0.2607        0.1365         -0.0653       3.0114                3.7398
P3    24     0.3866       0.4638               0.2616        0.0772         -0.1251       2.4260                3.4206
P4    11     0.3272       0.5641               0.3334        0.2369          0.0062       2.5748                2.5381
P5    34     0.3863       0.5246               0.3039        0.1383         -0.0824       2.4731                3.0828
P6    24     0.5148       0.5614               0.3313        0.0466         -0.1835       1.8692                2.7842
P7    28     0.4602       0.5475               0.3208        0.0873         -0.1394       2.0857                2.898

## 3. Le thème face au groupe témoin

`T1` contient les 366 entreprises SORT à poids égaux. `T1S` les repondère selon la répartition sectorielle des 134 entreprises retenues en 2026, fixe de 2000 à 2026. Les deux sont construits avec la composition de 2026 et les mêmes règles que le thème.

Deux limites bornent la lecture. D'abord, `P1` change de composition et ses poids dérivent, alors que les parts sectorielles de `T1S` restent fixes : l'écart `P1 - T1S` peut encore devoir une partie de sa valeur aux secteurs. Ensuite, partager le biais de survie ne veut pas dire en subir le même effet. L'écart est donc une comparaison descriptive entre deux univers de 2026 reconstitués dans le passé ; il ne mesure isolément ni le biais de survie ni l'effet de l'IA.

In [4]:
comparaison = pd.read_csv(SORTIE / "temoin_comparaison.csv")
print(comparaison[comparaison.periode == "complete"].set_index("serie")
      .drop(columns="periode").round(4).to_string())
print()

lignes = []
for temoin_nom in ["T1_reeq", "T1S_reeq"]:
    a, b = excedents(valeurs["P1_reeq"]), excedents(temoin[temoin_nom])
    a, b = a.align(b, join="inner")
    for nom, stat in [("sharpe", sharpe_annuel), ("volatilite", vol_annuelle),
                      ("rendement_excedentaire", lambda x: x.mean() * 252)]:
        res = risque.ecart_bootstrap(a.to_numpy(), b.to_numpy(), stat, BLOC, TIRAGES)
        lignes.append({"comparaison": f"P1_reeq - {temoin_nom}", "mesure": nom, **res})

face = pd.DataFrame(lignes)
face["significatif_5pct"] = face.p_bilaterale < 0.05
face.to_csv(SORTIE / "temoin_bootstrap.csv", index=False, encoding="utf-8")
print(face.round(4).to_string(index=False))

          annualise  volatilite  sharpe  sortino  repli_max   var99  perte_au_dela_99  beta_spy
serie                                                                                          
P1_reeq      0.1870      0.2173  0.8115   1.1626    -0.5184  0.0358            0.0510    1.0496
P1_cons      0.1878      0.2352  0.7698   1.0941    -0.5331  0.0424            0.0575    1.0992
T1_reeq      0.1608      0.1886  0.7854   1.1157    -0.5176  0.0324            0.0486    0.9248
T1_cons      0.1626      0.1963  0.7702   1.0894    -0.5325  0.0331            0.0496    0.9401
T1S_reeq     0.1627      0.1953  0.7738   1.1020    -0.5044  0.0342            0.0484    0.9474
T1S_cons     0.1577      0.2087  0.7163   1.0100    -0.5504  0.0362            0.0528    1.0004



       comparaison                 mesure  ecart     bas   haut  p_bilaterale  significatif_5pct
 P1_reeq - T1_reeq                 sharpe 0.0261 -0.1601 0.2038         0.814              False
 P1_reeq - T1_reeq             volatilite 0.0287  0.0223 0.0360         0.000               True
 P1_reeq - T1_reeq rendement_excedentaire 0.0282 -0.0081 0.0664         0.147              False
P1_reeq - T1S_reeq                 sharpe 0.0377 -0.1315 0.2008         0.697              False
P1_reeq - T1S_reeq             volatilite 0.0220  0.0170 0.0278         0.000               True
P1_reeq - T1S_reeq rendement_excedentaire 0.0252 -0.0099 0.0608         0.165              False


## 4. La contamination du témoin par les faux négatifs de la sélection

Le témoin rassemble les 366 entreprises que la règle de sélection a classées SORT. Une entreprise réellement exposée aux infrastructures de calcul mais écartée faute de preuve suffisante est un faux négatif : elle rapproche le témoin du thème et tend à réduire l'écart mesuré entre les deux. Le nombre de faux négatifs n'est pas connu.

Je ne peux pas les identifier, mais je peux isoler les dossiers les plus exposés à ce risque : les 93 entreprises classées SORT le 8 septembre 2026 par la consigne de clôture, parce que leur preuve restait insuffisante, alors qu'elles étaient auparavant `A_EXAMINER` ou `DOUTEUX`. Si la contamination pèse, retirer ces 93 entreprises doit éloigner le témoin du thème. Cette vérification est partielle : un faux négatif peut aussi se trouver parmi les 273 autres.

In [5]:
from src.construire_portefeuilles import lire_prix, simuler_groupe
from src.construire_temoin import (DEBUT, completer_seances, membres_temoin, statistiques)
from src.portefeuille import COUT, preparer_prix

brut = RACINE / "data" / "raw"
dates = pd.Index(pd.read_csv(brut / "calendrier_bourse.csv").date.astype(str), name="date")
dates = dates[dates >= DEBUT]
annuel = set(pd.Series(dates).groupby(pd.Series(dates).str[:4]).min())
fins = set(pd.Series(dates).groupby(pd.Series(dates).str[:7]).max())

decisions = pd.read_csv(RACINE / "data" / "review" / "decisions_selection.csv", dtype=str)
membres = membres_temoin(decisions)
fragiles = decisions[(decisions.verdict == "SORT") & (decisions.canal == "hors_perimetre_documente")]
print("dossiers SORT sur preuve insuffisante :", len(fragiles), fragiles.ancien_verdict.value_counts().to_dict())
print("secteurs :", fragiles.secteur.value_counts().to_dict())
print("services aux collectivites SORT au total :",
      int((decisions[decisions.verdict == "SORT"].secteur == "Utilities").sum()))

prepares = {}
for titre in membres.titre:
    prix, _ = completer_seances(lire_prix(brut / "prix_temoin" / f"{titre}.csv.gz"), dates)
    prepares[titre] = preparer_prix(prix).reindex(dates)
r_t = pd.DataFrame({t: p.rendement_valorisation for t, p in prepares.items()}, index=dates)
d_t = pd.DataFrame({t: p.dividende for t, p in prepares.items()}, index=dates)
dispo = pd.DataFrame({t: p.disponible for t, p in prepares.items()}, index=dates).fillna(False).astype(bool)
premiers = {t: dispo.index[dispo[t]][0] for t in dispo}

# Correlation de chaque titre temoin avec P1 reequilibre, selon la fragilite du dossier.
fragile = membres.set_index("titre").cik.isin(set(fragiles.cik))
correlation_p1 = r_t.corrwith(valeurs["P1_reeq"].pct_change())
print()
print(correlation_p1.groupby(fragile).agg(["count", "mean", "median"]).rename(
    index={False: "autres SORT", True: "SORT fragiles"}).round(3).to_string())

dossiers SORT sur preuve insuffisante : 93 {'A_EXAMINER': 71, 'DOUTEUX': 22}
secteurs : {'Industrials': 19, 'Financials': 14, 'Materials': 10, 'Health Care': 9, 'Energy': 9, 'Real Estate': 8, 'Consumer Staples': 7, 'Utilities': 7, 'Consumer Discretionary': 6, 'Communication Services': 3, 'Information Technology': 1}
services aux collectivites SORT au total : 7


               count   mean  median
cik                                
autres SORT      275  0.486   0.491
SORT fragiles     93  0.493   0.522


In [6]:
variantes = {}
for nom, groupe in [("T1_sans_fragiles", membres[~membres.cik.isin(set(fragiles.cik))]),
                    ("T1_fragiles", membres[membres.cik.isin(set(fragiles.cik))])]:
    rebuts = [[] for _ in range(5)]
    simuler_groupe(nom, groupe.assign(portefeuille=nom), dates, annuel, fins, dispo, premiers,
                   r_t, d_t, COUT, rebuts[0], variantes, *rebuts[1:])
variantes = pd.DataFrame(variantes, index=dates)

series = {"P1_reeq": valeurs["P1_reeq"], "T1_reeq": temoin["T1_reeq"],
          "T1_sans_fragiles_reeq": variantes["T1_sans_fragiles_reeq"],
          "T1_fragiles_reeq": variantes["T1_fragiles_reeq"]}
contamination = pd.DataFrame([statistiques(n, s.dropna(), sans_risque, marche) for n, s in series.items()])
contamination.to_csv(SORTIE / "temoin_contamination.csv", index=False, encoding="utf-8")
print(contamination.round(4).to_string(index=False))

p1 = contamination.set_index("serie").loc["P1_reeq"]
for t in ["T1_reeq", "T1_sans_fragiles_reeq"]:
    ligne = contamination.set_index("serie").loc[t]
    print(f"P1 - {t} : rendement {100 * (p1.annualise - ligne.annualise):+.2f} pts, "
          f"volatilite {100 * (p1.volatilite - ligne.volatilite):+.2f} pts, "
          f"beta {p1.beta_spy - ligne.beta_spy:+.3f}")

                serie  annualise  volatilite  sharpe  sortino  repli_max  var99  perte_au_dela_99  beta_spy
              P1_reeq     0.1870      0.2173  0.8115   1.1626    -0.5184 0.0358            0.0510    1.0496
              T1_reeq     0.1608      0.1886  0.7854   1.1157    -0.5176 0.0324            0.0486    0.9248
T1_sans_fragiles_reeq     0.1675      0.1901  0.8111   1.1561    -0.5142 0.0321            0.0480    0.9308
     T1_fragiles_reeq     0.1431      0.1905  0.6988   0.9850    -0.5325 0.0343            0.0507    0.9092
P1 - T1_reeq : rendement +2.63 pts, volatilite +2.87 pts, beta +0.125
P1 - T1_sans_fragiles_reeq : rendement +1.95 pts, volatilite +2.72 pts, beta +0.119


## 5. Ce que ce carnet change à la lecture des résultats

Les chiffres ci-dessus sont repris et interprétés dans la section XI de `research/resultats_risque.md`.

In [7]:
from src.empreintes import ecrire_manifeste

entrees = [SORTIE / "pipeline_portefeuilles.json", SORTIE / "risque_manifest.json",
           SORTIE / "temoin_manifest.json",
           RACINE / "data" / "raw" / "taux_sans_risque.csv",
           RACINE / "data" / "review" / "decisions_selection.csv",
           RACINE / "src" / "risque.py", RACINE / "src" / "construire_temoin.py"]
sorties = [SORTIE / n for n in ["risque_bootstrap_gestion.csv", "risque_correlation_ajustee.csv",
                                "temoin_bootstrap.csv", "temoin_contamination.csv"]]
manifeste = ecrire_manifeste(SORTIE / "robustesse_manifest.json", RACINE, entrees, sorties,
                             commande="src/robustesse_statistique.ipynb, execute en entier",
                             tirages=TIRAGES, bloc=BLOC)
print(len(manifeste["entrees_sha256"]), "entrees |", len(manifeste["sorties_sha256"]), "sorties")

7 entrees | 4 sorties
